# Retrieval-Augmented Generation with Milvus

This notebook demonstrates how to use Milvus for Retrieval-Augmented Generation (RAG).

In [ ]:
import os
from pymilvus import connections, utility
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.schema import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq.chat_models import ChatGroq
from langchain_milvus import Milvus
from langchain_community.document_loaders import RecursiveUrlLoader
from bs4 import BeautifulSoup
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.chains import create_retrieval_chain
from langchain_huggingface import HuggingFaceEmbeddings

# 定义常量
WEBSITE_URL = 'https://www.csusb.edu/its'
DATABASE_PATH = "milvus/jupyter_milvus_vector3.db"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"


def get_api_key():
    """从用户输入中获取 Groq API 密钥"""
    return input("Please enter your Groq API key: ")

def setup_vector_database(database_path):
    """建立 Milvus 数据库连接，并检查 collection 是否存在"""
    os.makedirs(os.path.split(database_path)[0], exist_ok=True)
    connections.connect("default", uri=database_path)
    return utility.has_collection("IT_support")

def create_embedding_model():
    """创建文本嵌入模型"""
    return HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

def clean_webpage_text(html_content):
    """清理 HTML 内容，提取可读文本"""
    soup = BeautifulSoup(html_content, 'html.parser')
    for element in soup(['script', 'style', 'header', 'footer', 'nav']):
        element.decompose()
    main_content = soup.find('main')
    text = main_content.get_text('\n') if main_content else soup.get_text('\n')
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return '\n'.join(lines)

def download_website_content():
    """下载并处理网站内容"""
    loader = RecursiveUrlLoader(
        url=WEBSITE_URL,
        prevent_outside=True,
        base_url=WEBSITE_URL
    )
    pages = loader.load()
    cleaned_pages = []
    for page in pages:
        clean_text = clean_webpage_text(page.page_content)
        cleaned_pages.append(Document(
            page_content=clean_text,
            metadata=page.metadata
        ))
    return cleaned_pages

def split_into_chunks(documents):
    """将文档拆分为较小的块"""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=300,
        is_separator_regex=False
    )
    return splitter.split_documents(documents)

def create_chatbot_prompt():
    """创建聊天机器人的指令模板"""
    system_message = """
    You are an AI assistant that provides answers strictly based on the provided context. Adhere to these guidelines:
     - Only answer questions based on the content within the <context> tags.
     - If the <context> doesn't contain relevant information, respond with: "I don't have enough information to answer this question."
     - Ask for clarification if questions are unclear.
     - Provide specific, concise answers with relevant statistics when available.
     - Don't add external information or make assumptions.
    """
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_message),
        ("human", "<question>{input}</question>\n\n<context>{context}</context>"),
    ])
    return prompt

def initialize_chatbot(api_key):
    """
    初始化聊天机器人：
    - 更新模型名称为支持的 "llama-2-70b-chat-hf"
    - 若检测到已存在的知识库，则删除旧 collection 以确保使用支持的 FLAT 索引类型
    """
    model = ChatGroq(
        model='deepseek-r1-distill-qwen-32b',
        temperature=0.9,
        api_key=api_key
    )
    
    if setup_vector_database(DATABASE_PATH):
        utility.drop_collection("IT_support")
        print("Existing collection dropped to re-create with supported index type.")
    
    print("Creating new knowledge base...")
    documents = download_website_content()
    chunks = split_into_chunks(documents)
    vector_store = Milvus.from_documents(
        documents=chunks,
        embedding=create_embedding_model(),
        collection_name="IT_support",
        connection_args={"uri": DATABASE_PATH},
        drop_old=True,
        index_params={"index_type": "FLAT", "metric_type": "L2", "params": {}}
    )
    
    retriever = vector_store.as_retriever(
        search_type="mmr",
        search_kwargs={
            "score_threshold": 1,
            "k": 3
        }
    )
    
    document_chain = create_stuff_documents_chain(
        model,
        create_chatbot_prompt()
    )
    
    return create_retrieval_chain(
        retriever,
        document_chain
    )

def format_sources(context_documents):
    """格式化返回结果中的来源 URL"""
    sources = []
    for i, doc in enumerate(context_documents):
        try:
            url = doc.metadata["source"]
            if url not in [s['url'] for s in sources]:
                sources.append({
                    'url': url,
                    'relevance_score': 1 / (i + 1)
                })
        except (IndexError, KeyError):
            continue
    sources.sort(key=lambda x: x['relevance_score'], reverse=True)
    source_list = "\n\nSources:\n" + "\n".join([
        f"{i+1}. {source['url']}" 
        for i, source in enumerate(sources[:4])
    ])
    return source_list

def answer_question(chain, question):
    """生成对用户提问的回答"""
    try:
        response = chain.invoke({"input": question})
        answer = response["answer"].strip()
        if "I don't have enough information to answer this question." in answer or \
           "You didn't ask a question." in answer or \
           "Hello" in answer or \
           "It appears that the question" in answer or \
           "Hello, it seems like you're looking for something related to the information provided in the context" in answer:
            return answer
        return answer + format_sources(response["context"])
    except Exception as e:
        if "503" in str(e) and "Service Unavailable" in str(e):
            return "I apologize, but the server is currently unavailable. This is a temporary issue. Please wait a few moments and try your question again."
        else:
            return f"I encountered an error while processing your question. Error details: {str(e)}"

def main():
    """主程序循环"""
    print(f"Welcome to the CSUSB ITS Support Chatbot!\nThis bot answers questions about: {WEBSITE_URL}")
    try:
        api_key = get_api_key()
        chain = initialize_chatbot(api_key)
        while True:
            question = input("\nEnter your question (or 'exit' to quit): ")
            if question.lower() == 'exit':
                print("Goodbye!")
                break
            answer = answer_question(chain, question)
            print("\nResponse:", answer)
    except KeyboardInterrupt:
        print("\nGoodbye!")
    except Exception as e:
        print(f"\nAn unexpected error occurred: {str(e)}")
        print("Please restart the program and try again.")

if __name__ == "__main__":
    main()  # 检查这个代码有没有问题


Welcome to the CSUSB ITS Support Chatbot!
This bot answers questions about: https://www.csusb.edu/its


Please enter your Groq API key:  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


Creating new knowledge base...



Enter your question (or 'exit' to quit):  hey



Response: <think>
Alright, so the user just sent "hey" as their question. I need to respond appropriately based on the context provided. 

First, I'll check the context. It's about IT services, including chatbots, support tickets, knowledge bases, software like Respondus, and contact information. 

Since the user's message is "hey", it's pretty vague. They might be greeting me or testing if I'm responsive. 

I should prompt them to ask a specific question. I need to keep the response friendly and open-ended to encourage them to provide more details. 

Looking at the guidelines, I should ask for clarification if the question is unclear. So, I'll mention that I can help with IT services and prompt them to clarify. 

Also, I should avoid adding external info, so I'll stick strictly to the provided context. 

I think the best response is to greet them and ask for clarification on how I can assist with IT services.
</think>

Hello! It seems like you're greeting me. How can I assist you wit


Enter your question (or 'exit' to quit):  do you know cusb



Response: <think>
Okay, the user is asking if I know about "cusb". Looking at the context provided, I see several mentions of "CSUSB" which stands for California State University, San Bernardino. The context includes information about the ITS department, email addresses, support contacts, and events like the annual ITS Tech Talks. 

The user might have a typo, so I should clarify if they meant CSUSB. Since the context doesn't mention "cusb" but does talk about CSUSB, I can safely assume they're referring to the university. I should explain that and offer to provide more specific information if they need it.
</think>

Yes, I am aware of "CSUSB," which stands for California State University, San Bernardino. The context provided discusses the university's Information Technology Services (ITS), including contacts, support information, and events like the annual ITS Tech Talks. If you have specific questions about CSUSB, feel free to ask!

Sources:
1. https://www.csusb.edu/its/events
2. ht


Enter your question (or 'exit' to quit):  how to apply csusb



Response: <think>
Okay, so I need to figure out how to apply to CSUSB based on the provided context. Let me go through the context step by step to see what information is available.

First, I see several sections like Major milestones, Configuration steps, Contact information, Policies & Standards, etc. However, most of these sections don't directly relate to the application process.

Looking through the context, I notice a section titled "CSU Apply" with a URL (csu-apply.edu). This seems relevant because it's likely the application portal. The context also mentions that this is where users without an established relationship can apply, which makes sense.

I don't see any other information related to application steps, deadlines, or requirements. The rest of the context talks about IT policies, support tickets, and other administrative details, which aren't helpful for applying.

So, based on the provided context, the only relevant information is the URL for CSU Apply. Therefore, the 